# COVID-19 Data Catalog
Convert this dataset and metadata into interactive HTML

In [2]:
import snowflake.connector
import pandas as pd

from IPython.core.display import display, HTML
from pandas_profiling import ProfileReport
from pathlib import Path

In [4]:
# Read connection configuration
home = str(Path.home()) + '/Documents/'
with open(home + 'snowflake.cfg') as f:
    config = f.readlines()

# Remove line returns    
config = [x.strip() for x in config]

# Set to local variables
sf_account = config[0]
sf_user = config[1]
sf_pwd = config[2]

In [5]:
# Connect to Snowflake
# https://uja84855.us-east-1.snowflakecomputing.com/console#/internal/worksheet

sf_warehouse = 'COMPUTE_WH'

ctx = snowflake.connector.connect(
    account = sf_account,
    user = sf_user,
    password = sf_pwd,
    warehouse = sf_warehouse
    # role = 'SYSADMIN'
    )
cs = ctx.cursor()

# Optional config when connecting ...
    #database = 'CIMBA_PROD',
    #schema = 'MCP'
# or after connecting ...
    # cs.execute("USE ROLE SYSADMIN;")
    # cs.execute("USE DATABASE CIMBA_PROD;")
    # cs.execute("USE WAREHOUSE CIMBA_WH;")

In [6]:
# Test the connection
print(type(ctx))
print(type(cs))

<class 'snowflake.connector.connection.SnowflakeConnection'>
<class 'snowflake.connector.cursor.SnowflakeCursor'>


In [7]:
ctx.role

'SYSADMIN'

# Get List of Tables

In [8]:
# Get metadata from this dedicated metadata table
table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."METADATA";'
df = pd.read_sql(table_query, ctx)

In [9]:
df.shape

(317, 7)

In [10]:
df.head()

,TABLE,DESCRIPTION,COLUMN,TYPE,NULLABLE,COMMENTS,SOURCE
0,CT_US_COVID_TESTS,US COVID-19 testing and mortality,None,None,None,None,None
1,CT_US_COVID_TESTS,ISO-3166-1 entity name,COUNTRY_REGION,VARCHAR(16777216),False,United States only,CovidTracking API
2,CT_US_COVID_TESTS,ISO-3166-2 entity name (U.S. State),PROVINCE_STATE,VARCHAR(16777216),True,United States only,CovidTracking API
3,CT_US_COVID_TESTS,Date of data point,DATE,DATE,False,None,CovidTracking API
4,CT_US_COVID_TESTS,Positive cases on date of report,POSITIVE,"NUMBER(38,0)",False,None,CovidTracking API


In [11]:
lst_tablenames_meta = df['TABLE'].unique()
lst_tablenames_meta = sorted(lst_tablenames_meta)
print(len(lst_tablenames_meta))
lst_tablenames_meta

22


['CT_US_COVID_TESTS',
 'DEMOGRAPHICS',
 'HDX_ACAPS',
 'HS_BULK_DATA',
 'HUM_RESTRICTIONS_AIRLINE',
 'HUM_RESTRICTIONS_COUNTRY',
 'IHME_COVID_19',
 'JHU_COVID_19',
 'KFF_HCP_CAPACITY',
 'KFF_US_ICU_BEDS',
 'KFF_US_POLICY_ACTIONS',
 'KFF_US_STATE_MITIGATIONS',
 'METADATA',
 'NYT_US_COVID19',
 'PCM_DPS_COVID19',
 'PCM_DPS_COVID19_DETAILS',
 'RKI_GER_DETAILED_DASHBOARD',
 'SCS_BE_DETAILED_HOSPITALISATIONS',
 'SCS_BE_DETAILED_MORTALITY',
 'SCS_BE_DETAILED_PROVINCE_CASE_COUNTS',
 'SCS_BE_DETAILED_TESTS',
 'WHO_SITUATION_REPORTS']

In [13]:
# select TABLE_NAME from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."TABLES" where TABLE_SCHEMA = 'PUBLIC';
table_query = 'select TABLE_NAME from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."TABLES" where TABLE_SCHEMA = \'PUBLIC\';'
df_infoschema = pd.read_sql(table_query, ctx)


In [14]:
# Get a sorted list of table names
lst_tablenames_info = df_infoschema['TABLE_NAME'].unique()
lst_tablenames_info = sorted(lst_tablenames_info)
print(len(lst_tablenames_info))
lst_tablenames_info

28


['CT_US_COVID_TESTS',
 'DATABANK_DEMOGRAPHICS',
 'DEMOGRAPHICS',
 'GOOG_GLOBAL_MOBILITY_REPORT',
 'HDX_ACAPS',
 'HS_BULK_DATA',
 'HUM_RESTRICTIONS_AIRLINE',
 'HUM_RESTRICTIONS_COUNTRY',
 'IHME_COVID_19',
 'JHU_COVID_19',
 'JHU_DASHBOARD_COVID_19_GLOBAL',
 'KFF_HCP_CAPACITY',
 'KFF_US_ICU_BEDS',
 'KFF_US_POLICY_ACTIONS',
 'KFF_US_STATE_MITIGATIONS',
 'METADATA',
 'NYC_HEALTH_TESTS',
 'NYT_US_COVID19',
 'NYT_US_REOPEN_STATUS',
 'PCM_DPS_COVID19',
 'PCM_DPS_COVID19_DETAILS',
 'RKI_GER_COVID19_DASHBOARD',
 'SCS_BE_DETAILED_HOSPITALISATIONS',
 'SCS_BE_DETAILED_MORTALITY',
 'SCS_BE_DETAILED_PROVINCE_CASE_COUNTS',
 'SCS_BE_DETAILED_TESTS',
 'VH_CAN_DETAILED',
 'WHO_SITUATION_REPORTS']

In [15]:
table_toc = ''
for table in lst_tablenames_info:
    
    table_toc += '<p class="tablemenu" onclick=\"displayContent('
    table_toc += ' \' ' + table + '\')">' + table + '</p>'
    table_toc += table + '</p>'
    table_toc += '\n'   

In [16]:
str(table_toc)

'<p class="tablemenu" onclick="displayContent( \' CT_US_COVID_TESTS\')">CT_US_COVID_TESTS</p>CT_US_COVID_TESTS</p>\n<p class="tablemenu" onclick="displayContent( \' DATABANK_DEMOGRAPHICS\')">DATABANK_DEMOGRAPHICS</p>DATABANK_DEMOGRAPHICS</p>\n<p class="tablemenu" onclick="displayContent( \' DEMOGRAPHICS\')">DEMOGRAPHICS</p>DEMOGRAPHICS</p>\n<p class="tablemenu" onclick="displayContent( \' GOOG_GLOBAL_MOBILITY_REPORT\')">GOOG_GLOBAL_MOBILITY_REPORT</p>GOOG_GLOBAL_MOBILITY_REPORT</p>\n<p class="tablemenu" onclick="displayContent( \' HDX_ACAPS\')">HDX_ACAPS</p>HDX_ACAPS</p>\n<p class="tablemenu" onclick="displayContent( \' HS_BULK_DATA\')">HS_BULK_DATA</p>HS_BULK_DATA</p>\n<p class="tablemenu" onclick="displayContent( \' HUM_RESTRICTIONS_AIRLINE\')">HUM_RESTRICTIONS_AIRLINE</p>HUM_RESTRICTIONS_AIRLINE</p>\n<p class="tablemenu" onclick="displayContent( \' HUM_RESTRICTIONS_COUNTRY\')">HUM_RESTRICTIONS_COUNTRY</p>HUM_RESTRICTIONS_COUNTRY</p>\n<p class="tablemenu" onclick="displayContent( \' 

# Generate Metadata

In [19]:
# Create an HTML metadata summary for each table

# Prevent truncating of strings from df
pd.set_option("display.max_colwidth", 10000) 

# Build HTML summary for each table
for table in lst_tablenames_meta:
    # print (table)
    tmp_html = ''
    tmp_html += '<!DOCTYPE html><html><head>'
    tmp_html += '<link rel="stylesheet" href="https://unpkg.com/purecss@2.0.3/build/pure-min.css" integrity="sha384-cg6SkqEOCV1NbJoCu11+bm0NvBRc8IYLRGXkmNrqUBfTjmMYwNKPWBTIKyw9mHNJ" crossorigin="anonymous">'
    tmp_html += '<meta name="viewport" content="width=device-width, initial-scale=1">'
    tmp_html += '</head>'
    tmp_html += '<body style="font-family:sans-serif;padding-left:20px;padding-right:20px;background-color:white">'
    tmp_html += '<h2>' + table + '</h2>'
    
    df_temp = df.loc[(df['TABLE'] == table) & (df['COLUMN'].isnull())]

    # Table description
    tmp_html += (df_temp['DESCRIPTION'].to_string(index=False)).lstrip()
    
    # Column metadata
    df_col = df.loc[(df['TABLE'] == table) & (df['COLUMN'].notnull())]
    df_col = df_col[['COLUMN', 'DESCRIPTION', 'COMMENTS', 'SOURCE']]
    df_col = pd.DataFrame(df_col, index=None)
    
    df_col.rename(columns = {'COLUMN':'COLUMN_NAME'}, inplace = True)
    tmp_html += df_col.to_html(justify='left', index=False, classes='pure-table pure-table-bordered')
    tmp_html += '<br><br>'
    tmp_html += '<hr>'
    
    # Finish the HTML
    tmp_html += '</body></html>'
    
    # Write HTML to file
    # 
    f = open( "content/metadata_" + table + ".html", "w+")
    f.write(tmp_html)
    f.close()

# Display to screen
# display(HTML(tmp_html))


# Generate Data Previews

In [21]:
# Create an HTML data preview for each table

# Prevent truncating of strings from df
pd.set_option("display.max_colwidth", 10000) 

# Build HTML summary for each table

for table in lst_tablenames_info:
    # print (table)
    tmp_html = ''
    tmp_html += '<!DOCTYPE html><html><head>'
    tmp_html += '<link rel="stylesheet" href="https://unpkg.com/purecss@2.0.3/build/pure-min.css" integrity="sha384-cg6SkqEOCV1NbJoCu11+bm0NvBRc8IYLRGXkmNrqUBfTjmMYwNKPWBTIKyw9mHNJ" crossorigin="anonymous">'
    tmp_html += '<meta name="viewport" content="width=device-width, initial-scale=1">'
    tmp_html += '</head>'
    tmp_html += '<body style="background-color:white;font-family:sans-serif;padding-left:20px;padding-right:20px">'
    tmp_html += '<h2>' + table + '</h2>'
    
    table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."' + table + '";'
    df_table = pd.read_sql(table_query, ctx)

    # First rows
    tmp_html += "<h3>First 5 Rows</h3>"
    df_temp = df_table.head(5)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')

    # Random sample of rows
    tmp_html += "<h3>Random 10 Rows</h3>"
    df_temp = df_table.sample(n=10)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')
    
    # Last rows
    tmp_html += "<h3>Last 5 Rows</h3>"
    df_temp = df_table.tail(5)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')
    
    tmp_html += '<br><br>'
    tmp_html += '<hr>'
    
    # Finish the HTML
    tmp_html += '</body></html>'
    
    # Write HTML to file
    # 
    f = open( "content/data_preview_" + table + ".html", "w+")
    f.write(tmp_html)
    f.close()

# Generate Data Profiles

In [93]:
# This function generates the data profile report and saves it to an HTML file
def generate_data_profile(df_table, table_name):
    
    filename = 'data_profile_' + table_name + ".html"
    rpt_title = 'Data Profile: ' + table_name.upper()
    
    myreport = ProfileReport(df_table, title=rpt_title, minimal=True)
    # myreport = ProfileReport(df_table, title=rpt_title)

#     myreport = ProfileReport(df_table, title="Pandas Profiling Report",

#         missing_diagrams={
#             "bar": True,
#             "matrix": False,
#             "heatmap": False,
#             "dendrogram": False                    
#         },

#         correlations={
#             "pearson":{
#                 "calculate": True,
#                 "warn_high_correlations": True,
#                 "threshold": 0.9
#             } ,
#             "spearman":{
#                 "calculate": False,
#                 "warn_high_correlations": False},
#             "kendall":{
#                 "calculate": False,
#                 "warn_high_correlations": False},
#             "phi_k":{
#                 "calculate": False,
#                 "warn_high_correlations": False},
#             "cramers":{
#                 "calculate": False,
#                 "warn_high_correlations": True,
#                 "threshold": 0.9},
#             "recoded":{
#                 "calculate": False,
#                 "warn_high_correlations": True,
#                 "threshold": 1.0}
#         },

#         samples={
#             "head": 3,
#             "tail": 3 
#         }
#     )

    myreport.to_file(filename)

In [94]:
# Create a data profile for each table and save to HTML

# Get list of tables
qry_tablenames = 'show tables in schema "STARSCHEMA_COVID19"."PUBLIC";'
results = cs.execute(qry_tablenames).fetchall()

i = 0
print('GENERATING DATA PROFILES for TABLES')
print('--------------------------')
for r in results:
    
    # NOTE: Exclude HS_BULK_DATA because too sparse
    if r[1] != "HS_BULK_DATA":
        if i >= 30:
            exit()
        else:
            df = None
            sf_table = r[1]
            # NOTE:  Use limit or sample to control how much to retrieve
            #table_query = 'select * from "COVID19"."PUBLIC"."' + sf_table + '" + ' limit 10,000;'
            table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."' + sf_table + '" sample (100 rows);'


            df = pd.read_sql(table_query, ctx)
            # print(df.head(3))

            print(r[1] + ' ' + str(df.shape))

            generate_data_profile(df, sf_table)
        
            i += 1

print('--------------------------')
print('Total Table Profiles: ' + str(len(results)))

GENERATING DATA PROFILES for TABLES
--------------------------
CT_US_COVID_TESTS (100, 31)



DATABANK_DEMOGRAPHICS (100, 11)



DEMOGRAPHICS (100, 10)



GOOG_GLOBAL_MOBILITY_REPORT (100, 14)



HDX_ACAPS (100, 15)


...


SCS_BE_DETAILED_PROVINCE_CASE_COUNTS (100, 11)



SCS_BE_DETAILED_TESTS (100, 3)



VH_CAN_DETAILED (100, 9)



WHO_SITUATION_REPORTS (100, 14)



--------------------------
Total Table Profiles: 27
